# CHARM — experiments on Google Colab
**Runtime → Change runtime type → GPU (T4 or better)**, then run the cells in order.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitrky/CHARM/blob/main/notebooks/CHARM_Colab.ipynb)

Everything is saved to Google Drive (`MyDrive/CHARM`), and every run is skipped if its result file already exists. If Colab disconnects, simply re-run all cells: finished runs are not repeated.

Phases (run in order; each can span several sessions):
1. **Core** — CHARM + the 4 most important baselines, seed 0 (enough to see whether the paper's claims hold)
2. **All baselines**, seeds 0–4
3. **Ablations**, seeds 0–4
4. **External validation, tables and figures**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/CHARM'
os.makedirs(BASE, exist_ok=True)


In [ ]:
%cd /content
!rm -rf CHARM && git clone -q https://github.com/iitrky/CHARM.git
%cd /content/CHARM
!pip install -q wfdb tabulate
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Download datasets (first session only; stored on Drive)

In [ ]:
DATA = f'{BASE}/data'
os.makedirs(DATA, exist_ok=True)
PTB = f'{DATA}/ptb-xl-1.0.3'
if not os.path.exists(f'{PTB}/ptbxl_database.csv'):
    !wget -q -O /content/ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
    !unzip -q /content/ptbxl.zip -d /content/ptbxl_tmp
    import glob, shutil
    src = glob.glob('/content/ptbxl_tmp/*')[0]
    shutil.move(src, PTB)
print(os.listdir(PTB)[:10])

In [ ]:
# External cohorts from the PhysioNet/CinC 2020 challenge (only needed for Phase 4)
CHAP, CPSC = f'{DATA}/chapman_shaoxing', f'{DATA}/cpsc_2018'
for name, path in [('chapman_shaoxing', CHAP), ('cpsc_2018', CPSC)]:
    if not os.path.exists(path):
        !wget -q -r -N -c -np -nH --cut-dirs=4 -P {DATA}/_c20 https://physionet.org/files/challenge-2020/1.0.2/training/{name}/
        !mv {DATA}/_c20/{name} {path} 2>/dev/null || find {DATA}/_c20 -maxdepth 3 -type d -name {name} -exec mv {{}} {path} \;
    print(name, sum(len(f) for _, _, f in os.walk(path)), 'files')
# If the folder layout differs, external_eval.py searches recursively for *.hea, so any nesting works.

## Smoke test (≈1 min) — must pass before anything else

In [ ]:
!python train_charm.py --synthetic --out /content/smoke
!python baselines.py --synthetic --model dagat --out /content/smoke
!python baselines.py --synthetic --model hyperfusion --out /content/smoke

## Runner (skips finished runs)

In [ ]:
import subprocess
R = f'{BASE}/results'
os.makedirs(R, exist_ok=True)
def run(script, tag, seed, extra=''):
    if os.path.exists(f'{R}/{tag}_seed{seed}.json'):
        print('skip', tag, seed); return
    cmd = f'python {script} --root {PTB} --seed {seed} --out {R} {extra}'
    print('RUN', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)
def charm(seed, tag='charm', extra=''):
    run('train_charm.py', tag, seed, f'--tag {tag} {extra}')
def base(seed, model, extra='', tag=None):
    run('baselines.py', tag or model, seed, f'--model {model} {extra}')

## Phase 1 — core comparison, seed 0
Check the output of the next cell before spending GPU time on Phase 2.

In [ ]:
charm(0)
for m in ['late_fusion', 'dagat', 'hyperfusion', 'xresnet']:
    base(0, m)

In [ ]:
import json
key = '12-lead|both|featmiss=0.0'
rows = []
for t in ['charm', 'late_fusion', 'dagat', 'hyperfusion', 'xresnet']:
    f = f'{R}/{t}_seed0.json'
    if os.path.exists(f):
        r = json.load(open(f))
        rows.append((t, r[key]['macro_auroc'], r['1-lead(I)|both|featmiss=0.0']['macro_auroc'],
                     r['12-lead|ecg_only|featmiss=0.0']['macro_auroc'], r[key].get('aurc'), r[key]['ece']))
print(f"{'model':14s} {'AUROC-12':>9s} {'AUROC-1lead':>12s} {'ECG-only':>9s} {'AURC':>7s} {'ECE':>7s}")
for t, a, b, c, d, e in rows:
    print(f"{t:14s} {a:9.3f} {b:12.3f} {c:9.3f} {d if d is None else round(d,4)!s:>7s} {e:7.3f}")
# Decision rule: CHARM should beat dagat/hyperfusion on the 1-lead and ECG-only columns and on AURC.
# If it does not, send these numbers to Claude before running Phase 2.

## Phase 2 — all baselines, seeds 0–4

In [ ]:
for s in range(5):
    charm(s)
    for m in ['xresnet', 'inception', 'cnn_lstm_att', 'transformer', 'ft_transformer',
              'early_fusion', 'late_fusion', 'medfuse', 'mc_dropout', 'dagat', 'hyperfusion']:
        base(s, m)
    base(s, 'late_fusion', '--temp_scale', tag='late_fusion_ts')

## Phase 3 — ablations, seeds 0–4

In [ ]:
ABL = {'abl_nofilm': '--no_film', 'abl_staticfilm': '--static_film', 'abl_noprior': '--no_prior',
       'abl_noimp': '--lam_imp 0', 'abl_nocons': '--lam_cons 0', 'abl_noproto': '--omega 1 --lam_proto 0',
       'abl_bce': '--bce_loss', 'abl_noaug': '--no_aug'}
for s in range(5):
    for tag, flags in ABL.items():
        charm(s, tag, flags)

## Phase 4 — external validation, tables, figures

In [ ]:
for s in range(5):
    for m in ['charm', 'xresnet', 'late_fusion', 'dagat', 'hyperfusion']:
        ck = f'{R}/{m}_seed{s}_final.pt'
        for name, path in [('chapman', CHAP), ('cpsc', CPSC)]:
            out = f'{R}/external_{name}_{m}_seed{s}.json'
            if os.path.exists(ck) and os.path.exists(path) and not os.path.exists(out):
                model_flag = '' if m == 'charm' else f'--model {m}'
                !python external_eval.py --ext_dir {path} --name {name} --ckpt {ck} --out {R} {model_flag}
!python aggregate.py --results {R} --ensemble late_fusion
!python figures.py --results {R} --root {PTB} --ckpt {R}/charm_seed0_final.pt
print('Send', R + '/ALL_TABLES.md', 'and the fig*.png files to Claude.')